# 03A Discrete Choice DP: The Rust (1987) Model

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/03-Economic-Modeling/03A_Discrete_Choice_DP_Rust.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=03-Economic-Modeling/03A_Discrete_Choice_DP_Rust.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)


In [1]:
# === Environment Setup ===
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Configuration ---


### Table of Contents
1. [The Lens: Discrete Choices with Conditional Value Functions](#the-lens-discrete-choices-with-conditional-value-functions)
2. [The Rust (1987) Model: Optimal Replacement](#1-the-rust-1987-model-optimal-replacement)
3. [Summary](#summary)

> **Historical Context — Harold Zurcher's buses (1987).** John Rust's 1987 Econometrica paper modeled a Madison, Wisconsin bus mechanic replacing engines optimally under uncertainty — the first landmark pairing of dynamic programming with micro panel data. Structural estimation as practiced across IO, labor, and macro largely traces to this study of grease and mileage.

## The Lens: Discrete Choices with Conditional Value Functions
**What problem are we solving?**
Many economic decisions are discrete: replace or keep, work or retire, default or repay. These choices create kinks in the value function that break smooth optimization methods. We need a way to evaluate each discrete alternative while still accounting for optimal future behavior.

**Why this method?**
The **conditional value function** approach computes the value of each discrete choice separately and then takes the upper envelope. This makes discrete-choice dynamic programs tractable and leads to clear policy thresholds, as in Rust’s (1987) bus replacement model.



**Economic question.** In *03A Discrete Choice DP: The Rust (1987) Model*, what must remain economically invariant when the computational representation changes? Dynamic models convert economic incentives into a recursive computational object. The practical question is how an agent's current choice changes both today's payoff and tomorrow's feasible states. A trustworthy solution should make that mechanism visible through the Bellman equation, the policy rule, and a diagnostic such as a residual or Euler-equation error. Keep the economic state, control, transition law, and numerical approximation conceptually separate.

### Learning Objectives
* **Formulate** discrete-choice dynamic programs using conditional value functions.
* **Implement** value function iteration for the Rust optimal replacement model.
* **Interpret** replacement thresholds and policy functions.

### Prerequisites
* **`03-Economic-Modeling/01_Dynamic_Programming.ipynb`**: Bellman equation and VFI basics.
* **`02-Numerical-Methods/05_Optimization.ipynb`**: Optimization primitives.
* **Economics:** Discrete choice logic and dynamic incentives.
* **Learning-path prerequisite:** [`02_DP_with_Continuous_States.ipynb`](02_DP_with_Continuous_States.ipynb)


> **Learning path:** Building on [`02_DP_with_Continuous_States.ipynb`](02_DP_with_Continuous_States.ipynb); next continue with [`03B_Continuous_State_DP_Interpolation.ipynb`](03B_Continuous_State_DP_Interpolation.ipynb).


### 1. The Rust (1987) Model: Optimal Replacement

Rust's model provides a canonical example of a discrete choice DP problem. A manager, Harold Zurcher, must decide each period whether to replace a bus engine (`d=1`, a discrete choice) or to keep it and perform maintenance (`d=0`).

- **State Variable:** The state of the system is the odometer reading, `x`, which represents the engine's mileage.
- **Flow Utility:** The utility in a period depends on the maintenance cost, which is a function of mileage, and the operating costs.
- **Transition:** The mileage `x` evolves stochastically over time.

The manager's problem is to choose a sequence of replacement and maintenance decisions to minimize the total expected discounted cost over the infinite horizon.


#### The Conditional Value Function

The key to solving these models is to decompose the value function. The overall value function is:
$$ V(x) = \max_{d \in \{0, 1\}} \{ v(x, d) \} $$
where $v(x, d)$ is the **conditional value function**—the value of committing to the discrete choice `d`.

1.  **Value of Replacing (`d=1`):** If the manager replaces the engine, he pays a fixed replacement cost `RC` and gets a new engine with zero mileage. The value is:
    $$ v(x, 1) = -RC + \beta E[V(0)] $$

2.  **Value of Keeping (`d=0`):** If the manager keeps the engine, he pays maintenance cost $c(x)$ and moves to a new state $x'$.
    $$ v(x, 0) = -c(x) + \beta E[V(x') | x] $$

This is a Bellman fixed-point calculation. A **nested fixed point** estimator adds an outer parameter-optimization loop around this solution step. The code below is a simplified replacement problem with no unobserved taste shocks and a deterministic reset to zero; it is not a replication of the estimated Rust model.


In [ ]:
class OptimalReplacement:
    """
    A class to solve the Rust (1987) optimal replacement problem using 
    Value Function Iteration.
    """
    def __init__(self, beta=0.9, RC=10, c_scale=0.01, max_mileage=100, n_states=100):
        self.beta = beta          # Discount factor
        self.RC = RC              # Replacement cost
        self.c_scale = c_scale    # Cost scale parameter
        self.n_states = n_states  # Number of grid points
        self.state_grid = np.arange(n_states)

        # Transition probability: mileage increases by 0, 1, or 2 units
        self.p = [0.1, 0.7, 0.2]
        self.P = self._build_transition_matrix()

        # Cost function: linear in mileage
        self.cost = self.c_scale * self.state_grid

    def _build_transition_matrix(self):
        """Constructs the transition matrix P[x, x']."""
        P = np.zeros((self.n_states, self.n_states))
        for i in range(self.n_states):
            for k, prob in enumerate(self.p):
                if i + k < self.n_states:
                    P[i, i+k] = prob
            # Absorbing state at the boundary
            P[i, -1] += 1 - np.sum(P[i, :])
        return P

    def solve(self, tol=1e-6, max_iter=1000):
        """Solves the model using Value Function Iteration."""
        V = np.zeros(self.n_states)

        for i in range(max_iter):
            # 1. Expected Value of future state
            EV = self.P @ V

            # 2. Conditional Value Functions
            # Value of Keeping (d=0)
            v_keep = -self.cost + self.beta * EV

            # Value of Replacing (d=1)
            v_replace = -self.RC + self.beta * V[0]

            # 3. Bellman Operator
            V_new = np.maximum(v_keep, v_replace)

            if np.max(np.abs(V_new - V)) < tol:
                return V_new, v_keep, v_replace
            V = V_new

        return V, v_keep, v_replace

model = OptimalReplacement(RC=8)
V, v_keep, v_replace = model.solve()

# Find the threshold state where optimal choice switches from Keep to Replace
policy = (v_replace > v_keep).astype(int)
replacement_states = np.flatnonzero(policy)
threshold_state = int(replacement_states[0]) if replacement_states.size else None
print(f"Optimal replacement threshold: {threshold_state if threshold_state is not None else 'no replacement on this grid'}")


In [ ]:
### Visualizing the Optimal Policy

plt.figure(figsize=(10, 6))
plt.plot(model.state_grid, v_keep, label='Value of Keeping Engine', linewidth=2)
plt.plot(model.state_grid, np.full(model.n_states, v_replace, dtype=float), label='Value of Replacing', linestyle='--', linewidth=2)
if threshold_state is not None:
    plt.axvline(x=threshold_state, color='red', linestyle=':', label=f'Replacement Threshold (x={threshold_state})')

plt.title('Optimal Replacement Policy: Keep vs. Replace')
plt.xlabel('Mileage (State x)')
plt.ylabel('Value Function')
plt.legend()
plt.show()


## Key Equations

These relations are collected from the derivations above as a review map. Their assumptions and derivations remain part of the result; this box is not a substitute for them.

**1. Core relation**

$$V(x) = \max_{d \in \{0, 1\}} \{ v(x, d) \}$$

**2. Core relation**

$$v(x, 1) = -RC + \beta E[V(0)]$$

**3. Core relation**

$$v(x, 0) = -c(x) + \beta E[V(x') | x]$$


## Hotz-Miller CCP Inversion: Avoiding a Full Inner Value Iteration

Hotz and Miller (1993) show that in dynamic discrete-choice models with additive Type-I extreme-value shocks, observed conditional choice probabilities (CCPs) reveal differences in choice-specific value functions. For two actions, normalize action 0 as the reference. The logit CCP is

$$
P_1(s)=\frac{e^{v_1(s)}}{e^{v_0(s)}+e^{v_1(s)}}.
$$

Taking odds and logs gives the inversion directly:

$$
\boxed{v_1(s)-v_0(s)=\log P_1(s)-\log P_0(s).}
$$

This is computationally important because a first-stage CCP estimate can replace repeated full backward-induction solves inside some estimation routines. The inversion does **not** identify structural parameters by itself: it inherits the maintained shock distribution, state-transition specification, and first-stage CCP quality.


In [4]:
# Numerical sanity check of the binary Hotz-Miller inversion.
ccp_replace = np.array([0.08, 0.15, 0.32, 0.55, 0.78])
ccp_keep = 1.0 - ccp_replace
value_difference = np.log(ccp_replace) - np.log(ccp_keep)
recovered_ccp = 1.0 / (1.0 + np.exp(-value_difference))
assert np.allclose(recovered_ccp, ccp_replace, atol=1e-12)

pd.DataFrame({
    "P(replace|s)": ccp_replace,
    "v_replace - v_keep": value_difference,
    "recovered CCP": recovered_ccp,
})


,P(replace|s),v_replace - v_keep,recovered CCP
0,0.08,-2.442347,0.08
1,0.15,-1.734601,0.15
2,0.32,-0.753772,0.32
3,0.55,0.200671,0.55
4,0.78,1.265666,0.78


**Reference:** Hotz, V. J., & Miller, R. A. (1993). Conditional choice probabilities and the estimation of dynamic models. *Review of Economic Studies*, 60(3), 497–529.


## Exercises

**1. Mechanism and assumptions (Conceptual):** State the equilibrium/optimality condition that organizes **03A Discrete Choice DP: The Rust (1987) Model**. Explain which assumption guarantees existence, uniqueness, or stability, and identify a limiting case where that argument weakens.

**2. Reproduce and diagnose (Applied):** Reproduce one quantitative result from the sections on 1. The Rust (1987) Model: Optimal Replacement, The Conditional Value Function. Change one economically meaningful parameter over a defensible grid, report the policy/value/equilibrium response, and verify convergence with a residual or tighter tolerance.

**3. Robust extension (Challenge):** Design a policy or shock counterfactual that changes one mechanism at a time. Compare welfare or transition dynamics against the baseline and explain which conclusion is structural versus calibration-specific.

**3b. Failure analysis (Challenge):** The MLE of the Rust model converges to a boundary (a transition probability of exactly 0) with a flat likelihood in that direction and huge standard errors. Diagnose the identification/boundary problem, repair with a reparameterization (logit-type probabilities) or penalization, and inspect the profile likelihood to show the flatness.

<details>
<summary>Solution guidance</summary>

A strong solution states assumptions before computation, includes an independent diagnostic or limiting-case check, and interprets the result in the units of the economic problem. For the challenge, separate changes caused by the economic assumption from changes caused by numerical approximation or tuning.

</details>


# Summary

This part showed how discrete choices create nested value functions and how Rust-style optimal replacement problems are solved using conditional value functions.


## References & Further Reading

- Stokey, N. L., Lucas, R. E. Jr. & Prescott, E. C. (1989). *Recursive Methods in Economic Dynamics*. Harvard University Press.
- Ljungqvist, L. & Sargent, T. J. (2018). *Recursive Macroeconomic Theory* (4th ed.). MIT Press.
- Rust, J. (1987). Optimal replacement of GMC bus engines: An empirical model of Harold Zurcher. *Econometrica*, 55(5), 999–1033.
